In [ ]:
import os, gc, glob
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import find_peaks, savgol_filter, resample
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit

try:
    import mat73
except ImportError:
    print("pip install mat73  (required for PulseDB only)")
    mat73 = None

# ============================================================
# CONFIGURATION
# ============================================================
FS_UCI      = 125
FS_HOSP     = 1000
FS_SENSORS  = 125
FS_BCG      = 125
FS_PULSE    = 125
FIXED_LEN   = 128
MIN_BEAT_S  = 0.40
MAX_BEAT_S  = 1.50

# ── Population-level normalization constants (NO per-beat leakage) ──
# ABP stored = (abp_mmhg - POP_MU) / POP_SIG
# Derived from UCI training set statistics (mmHg)
POP_MU  = 66.2  # ≈ population mean DBP
POP_SIG = 23.6   # ≈ population SBP range / 2  (covers ~40–180 mmHg)

# Input paths
PHASE0_UCI_DATA  = "outputs/phase_0/data/uci_all_parts.npz"
PHASE0_HOSP_NPZ  = "outputs/phase_0/data/hospital_matched.npz"
PHASE0_HOSP_CSV  = "outputs/phase_0/data/hospital_matched.csv"
SENSORS_DIR      = "datasets/sensors_dataset"
BCG_DIR          = "datasets/bcg_dataset"
PULSEDB_DIR      = "datasets/PulseDB_Vital"

OUT_DIR          = "outputs/phase_1/data"
PULSEDB_CHUNK_DIR= os.path.join(OUT_DIR, "pulsedb_chunks")
PULSEDB_LOG      = os.path.join(PULSEDB_CHUNK_DIR, "processed_files.txt")
SAVE_EVERY_N     = 50

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PULSEDB_CHUNK_DIR, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================
def normalize_ppg(x):
    """Min-max normalize PPG to [0,1]"""
    d = x.max() - x.min()
    return (x - x.min()) / d if d > 1e-8 else np.zeros_like(x)

def normalize_abp_population(abp_mmhg):
    """
    Population-level normalization — NO per-beat BP label leakage.
    Phase 3 denorm: abp_mmhg = abp_stored * POP_SIG + POP_MU
    """
    return ((abp_mmhg - POP_MU) / POP_SIG).astype(np.float32)

def qc_bp(sbp, dbp):
    """Return True if BP passes clinical QC"""
    pp = sbp - dbp
    return pp >= 15 and sbp <= 220 and dbp >= 40

def get_mat_array(filepath):
    mat = sio.loadmat(filepath)
    best_key, best_data, max_size = None, None, -1
    for k, v in mat.items():
        if k.startswith('__') or v.dtype.kind in {'U', 'S'}:
            continue
        try:
            arr = (np.array([np.squeeze(d) for d in v], dtype=np.float64)
                   if v.dtype == object else np.squeeze(v).astype(np.float64))
            if arr.size > max_size:
                max_size, best_key, best_data = arr.size, k, arr
        except Exception:
            continue
    if best_data is None:
        raise ValueError(f"No signal found in {filepath}")
    return best_key, best_data

# ============================================================
# 1. UCI
# ============================================================
def extract_uci():
    print("\n" + "="*55)
    print("PHASE 1 [FIXED]: UCI")
    print("="*55)
    if not os.path.exists(PHASE0_UCI_DATA):
        print(f"  [ERROR] Not found: {PHASE0_UCI_DATA}"); return

    raw   = np.load(PHASE0_UCI_DATA)
    X_raw = raw["X"]
    out_ppg, out_abp, out_bp, out_dur, out_subj, out_peak = [], [], [], [], [], []

    for i in tqdm(range(len(X_raw)), desc="  UCI"):
        ppg_full, abp_full = X_raw[i, 0], X_raw[i, 1]
        try:
            valleys, _ = find_peaks(
                -savgol_filter(ppg_full, 11, 3),
                distance=int(FS_UCI * MIN_BEAT_S), prominence=0.05)
        except Exception:
            continue
        if len(valleys) < 2: continue

        for j in range(len(valleys) - 1):
            s, e   = valleys[j], valleys[j+1]
            beat_s = (e - s) / FS_UCI
            if not (MIN_BEAT_S <= beat_s <= MAX_BEAT_S): continue

            beat_abp = abp_full[s:e]
            sbp, dbp = float(beat_abp.max()), float(beat_abp.min())
            if not qc_bp(sbp, dbp): continue

            ppg_r = normalize_ppg(resample(ppg_full[s:e], FIXED_LEN))
            abp_r = normalize_abp_population(resample(beat_abp, FIXED_LEN))

            out_ppg.append(ppg_r.astype(np.float32))
            out_abp.append(abp_r)
            out_bp.append([sbp, dbp])
            out_dur.append(np.float32(beat_s))
            out_subj.append(np.int32(i))
            out_peak.append(np.int32(np.argmax(ppg_r)))

    if out_ppg:
        X_ppg = np.expand_dims(np.stack(out_ppg), 1)
        np.savez_compressed(
            os.path.join(OUT_DIR, "universal_tensors.npz"),
            X_ppg=X_ppg, Y_abp=np.stack(out_abp),
            Y_bp=np.array(out_bp), beat_durations=np.array(out_dur),
            subject_ids=np.array(out_subj), peak_indices=np.array(out_peak),
            pop_mu=np.float32(POP_MU), pop_sig=np.float32(POP_SIG))
        print(f"  -> Saved {len(X_ppg):,} UCI beats  |  norm: pop(mu={POP_MU},sig={POP_SIG})")

# ============================================================
# 2. Hospital (no Y_abp — PPG only)
# ============================================================
def extract_hospital():
    print("\n" + "="*55)
    print("PHASE 1 [FIXED]: Hospital")
    print("="*55)
    if not (os.path.exists(PHASE0_HOSP_NPZ) and os.path.exists(PHASE0_HOSP_CSV)):
        print("  [ERROR] Hospital files not found"); return

    X_raw   = np.load(PHASE0_HOSP_NPZ)["X"]
    df_meta = pd.read_csv(PHASE0_HOSP_CSV)
    out_ppg, out_bp, out_dur, out_subj, out_peak = [], [], [], [], []

    for i in tqdm(range(len(X_raw)), desc="  Hospital"):
        sbp      = float(df_meta.loc[i, "Systolic Blood Pressure(mmHg)"])
        dbp      = float(df_meta.loc[i, "Diastolic Blood Pressure(mmHg)"])
        subj_id  = int(df_meta.loc[i, "subject_ID"])
        if not qc_bp(sbp, dbp): continue

        ppg_full = X_raw[i]
        try:
            valleys, _ = find_peaks(
                -savgol_filter(ppg_full, 51, 3),
                distance=int(FS_HOSP * MIN_BEAT_S), prominence=0.02)
        except Exception:
            continue
        if len(valleys) < 2: continue

        for j in range(len(valleys) - 1):
            s, e   = valleys[j], valleys[j+1]
            beat_s = (e - s) / FS_HOSP
            if not (MIN_BEAT_S <= beat_s <= MAX_BEAT_S): continue

            ppg_r = normalize_ppg(resample(ppg_full[s:e], FIXED_LEN))
            out_ppg.append(ppg_r.astype(np.float32))
            out_bp.append([sbp, dbp])
            out_dur.append(np.float32(beat_s))
            out_subj.append(np.int32(subj_id))
            out_peak.append(np.int32(np.argmax(ppg_r)))

    if out_ppg:
        X_ppg = np.expand_dims(np.stack(out_ppg), 1)
        # Hospital has no ABP waveform — Phase 3 uses get_medical_mu_sig fallback
        np.savez_compressed(
            os.path.join(OUT_DIR, "hospital_tensors.npz"),
            X_ppg=X_ppg, Y_bp=np.array(out_bp),
            beat_durations=np.array(out_dur),
            subject_ids=np.array(out_subj), peak_indices=np.array(out_peak),
            pop_mu=np.float32(POP_MU), pop_sig=np.float32(POP_SIG))
        print(f"  -> Saved {len(X_ppg):,} Hospital beats  |  no Y_abp (PPG-only dataset)")

# ============================================================
# 3. Sensors
# ============================================================
def _mat_str_col(col):
    """MATLAB char/cell column -> list[str]. Handles nested 1-element ndarrays."""
    out = []
    for x in np.array(col).ravel():
        while isinstance(x, np.ndarray) and x.size:
            x = x.ravel()[0]
        out.append(str(x))
    return out


def extract_sensors():
    print("\n" + "="*55)
    print("PHASE 1 [FIXED]: Sensors")
    print("="*55)
    out_ppg, out_abp, out_bp, out_dur, out_subj, out_peak, out_fold = [], [], [], [], [], [], []
    _SENS_PID = {}   # real patient string -> contiguous int, stable across folds

    for fold in range(5):
        mat_path = os.path.join(SENSORS_DIR, f"signal_mabp_fold_{fold}.mat")
        if not os.path.exists(mat_path): continue
        mat = sio.loadmat(mat_path)
        ppg_data = mat['signal']
        abp_data = mat['abp_signal']
        sbp_data = mat['SP']
        dbp_data = mat['DP']
        # REAL patient identifiers ship with this corpus. The previous version keyed
        # subject_ids on the ROW INDEX, which invented ~2.4k pseudo-subjects out of
        # 1,195 real patients and split every patient's rows across folds.
        pat_col  = _mat_str_col(mat['patient'])
        track    = {"ok": 0, "qc": 0, "peak": 0}

        for i in tqdm(range(len(ppg_data)), desc=f"  Sensors fold {fold}"):
            sbp = float(sbp_data[i][0])
            dbp = float(dbp_data[i][0])
            if not qc_bp(sbp, dbp): track["qc"] += 1; continue

            ppg_full = np.squeeze(np.array(ppg_data[i], dtype=np.float64))
            abp_beat = np.squeeze(np.array(abp_data[i],  dtype=np.float64))
            if ppg_full.ndim != 1 or ppg_full.size < 100: continue

            abp_r = normalize_abp_population(resample(abp_beat, FIXED_LEN))

            try:
                valleys, _ = find_peaks(
                    -savgol_filter(ppg_full, 11, 3),
                    distance=int(FS_SENSORS * MIN_BEAT_S), prominence=0.05)
            except Exception:
                continue
            if len(valleys) < 2: track["peak"] += 1; continue

            for j in range(len(valleys) - 1):
                s, e   = valleys[j], valleys[j+1]
                beat_s = (e - s) / FS_SENSORS
                if not (MIN_BEAT_S <= beat_s <= MAX_BEAT_S): continue
                ppg_r = normalize_ppg(resample(ppg_full[s:e], FIXED_LEN))
                out_ppg.append(ppg_r.astype(np.float32))
                out_abp.append(abp_r)
                out_bp.append([sbp, dbp])
                out_dur.append(np.float32(beat_s))
                out_subj.append(np.int32(_SENS_PID.setdefault(pat_col[i], len(_SENS_PID))))
                out_peak.append(np.int32(np.argmax(ppg_r)))
                out_fold.append(np.int8(fold))
                track["ok"] += 1

        print(f"  Fold {fold}: saved={track['ok']}  qc_drop={track['qc']}  no_peak={track['peak']}")

    print(f"  -> {len(_SENS_PID):,} REAL patients (row-index keying would have given ~2,383)")

    if out_ppg:
        X_ppg = np.expand_dims(np.stack(out_ppg), 1)
        np.savez_compressed(
            os.path.join(OUT_DIR, "sensors_tensors.npz"),
            subject_unit=np.str_("patient"),
            X_ppg=X_ppg, Y_abp=np.stack(out_abp),
            Y_bp=np.array(out_bp), beat_durations=np.array(out_dur),
            subject_ids=np.array(out_subj), peak_indices=np.array(out_peak),
            fold_indices=np.array(out_fold),
            pop_mu=np.float32(POP_MU), pop_sig=np.float32(POP_SIG))
        print(f"  -> Saved {len(X_ppg):,} Sensors beats")

# ============================================================
# 4. BCG
# ============================================================
def extract_bcg():
    print("\n" + "="*55)
    print("PHASE 1 [FIXED]: BCG")
    print("="*55)
    out_ppg, out_abp, out_bp, out_dur, out_subj, out_peak, out_fold = [], [], [], [], [], [], []

    for fold in range(5):
        mat_path = os.path.join(BCG_DIR, f"signal_mabp_fold_{fold}.mat")
        if not os.path.exists(mat_path):
            print(f"  [WARNING] Not found: {mat_path}"); continue
        mat = sio.loadmat(mat_path)
        ppg_data = mat['signal']
        abp_data = mat['abp_signal']
        sbp_data = mat['SP']
        dbp_data = mat['DP']
        track    = {"ok": 0, "qc": 0, "peak": 0}

        for i in tqdm(range(len(ppg_data)), desc=f"  BCG fold {fold}"):
            sbp = float(sbp_data[i][0])
            dbp = float(dbp_data[i][0])
            if not qc_bp(sbp, dbp): track["qc"] += 1; continue

            ppg_full = np.squeeze(np.array(ppg_data[i], dtype=np.float64))
            abp_beat = np.squeeze(np.array(abp_data[i],  dtype=np.float64))
            if ppg_full.ndim != 1 or ppg_full.size < 100: continue

            abp_r = normalize_abp_population(resample(abp_beat, FIXED_LEN))

            try:
                valleys, _ = find_peaks(
                    -savgol_filter(ppg_full, 11, 3),
                    distance=int(FS_BCG * MIN_BEAT_S), prominence=0.05)
            except Exception:
                continue
            if len(valleys) < 2: track["peak"] += 1; continue

            for j in range(len(valleys) - 1):
                s, e   = valleys[j], valleys[j+1]
                beat_s = (e - s) / FS_BCG
                if not (MIN_BEAT_S <= beat_s <= MAX_BEAT_S): continue
                ppg_r = normalize_ppg(resample(ppg_full[s:e], FIXED_LEN))
                out_ppg.append(ppg_r.astype(np.float32))
                out_abp.append(abp_r)
                out_bp.append([sbp, dbp])
                out_dur.append(np.float32(beat_s))
                # This corpus ships `patient` and `trial` REDACTED to the literal 'X',
                # so no patient identifier exists. The previous version keyed on the row
                # index `i`, which RESTARTS every fold: rows 0 of folds 0..4 collapsed
                # into one pseudo-subject spanning five unrelated recordings (SBP 102 to
                # 155 mmHg for "subject 0"). We key on (fold, row) so each recording is
                # its own unit, and mark the unit as RECORDING, not patient.
                out_subj.append(np.int32(fold * 100_000 + i))
                out_peak.append(np.int32(np.argmax(ppg_r)))
                out_fold.append(np.int8(fold))
                track["ok"] += 1

        print(f"  Fold {fold}: saved={track['ok']}  qc_drop={track['qc']}  no_peak={track['peak']}")

    print("  -> unit = RECORDING (patient IDs are redacted in this corpus)")

    if out_ppg:
        X_ppg = np.expand_dims(np.stack(out_ppg), 1)
        np.savez_compressed(
            os.path.join(OUT_DIR, "bcg_tensors.npz"),
            subject_unit=np.str_("recording"),
            X_ppg=X_ppg, Y_abp=np.stack(out_abp),
            Y_bp=np.array(out_bp), beat_durations=np.array(out_dur),
            subject_ids=np.array(out_subj), peak_indices=np.array(out_peak),
            fold_indices=np.array(out_fold),
            pop_mu=np.float32(POP_MU), pop_sig=np.float32(POP_SIG))
        print(f"  -> Saved {len(X_ppg):,} BCG beats")

# ============================================================
# 5. PulseDB (chunked, memory-safe, auto-resume)
# ============================================================
def extract_pulsedb():
    print("\n" + "="*55)
    print("PHASE 1 [FIXED]: PulseDB")
    print("="*55)
    if mat73 is None:
        print("  [SKIP] mat73 not installed"); return

    all_files = glob.glob(os.path.join(PULSEDB_DIR, "*.mat"))
    if not all_files:
        print(f"  [ERROR] No .mat files in {PULSEDB_DIR}"); return

    done = set(open(PULSEDB_LOG).read().splitlines()) if os.path.exists(PULSEDB_LOG) else set()
    todo = [f for f in all_files if os.path.basename(f) not in done]
    print(f"  Total={len(all_files):,}  Done={len(done):,}  Remaining={len(todo):,}")
    if not todo: print("  All done!"); return

    out_ppg, out_abp, out_bp, out_dur, out_subj = [], [], [], [], []
    out_age, out_gender, out_height, out_weight  = [], [], [], []
    batch_names = []
    track  = {"ok": 0, "qc": 0, "peak": 0, "err": 0}
    chunk_idx = len(glob.glob(os.path.join(PULSEDB_CHUNK_DIR, "pulsedb_part_*.npz"))) + 1

    def _flush(is_last):
        nonlocal chunk_idx
        if not out_ppg: return
        cname = f"pulsedb_part_{chunk_idx:04d}.npz"
        X_ppg = np.expand_dims(np.stack(out_ppg), 1)
        np.savez_compressed(
            os.path.join(PULSEDB_CHUNK_DIR, cname),
            X_ppg=X_ppg, Y_abp=np.stack(out_abp),
            Y_bp=np.array(out_bp), beat_durations=np.array(out_dur),
            subject_ids=np.array(out_subj),
            age=np.array(out_age), gender=np.array(out_gender),
            height=np.array(out_height), weight=np.array(out_weight),
            pop_mu=np.float32(POP_MU), pop_sig=np.float32(POP_SIG))
        print(f"  Saved {cname}  ({len(X_ppg):,} beats)")
        chunk_idx += 1
        with open(PULSEDB_LOG, "a") as f:
            f.writelines(n + "\n" for n in batch_names)
        batch_names.clear()
        for lst in [out_ppg, out_abp, out_bp, out_dur, out_subj,
                    out_age, out_gender, out_height, out_weight]:
            lst.clear()
        gc.collect()

    def _parse(v, idx, is_gender=False):
        try:
            val = v[idx] if hasattr(v, '__len__') and len(v) > idx else v
            val = float(np.squeeze(val))
            if is_gender:
                return 1.0 if val == 1.0 else 0.0
            return val
        except Exception:
            return np.nan

    for fi, fpath in enumerate(todo):
        fname = os.path.basename(fpath)
        batch_names.append(fname)
        try:
            data = mat73.loadmat(fpath)
            wins = data.get('Subj_Wins', {})
            if not wins or 'PPG_F' not in wins or 'ABP_Raw' not in wins:
                track["err"] += 1; continue

            ppg_list = wins['PPG_F']
            abp_list = wins['ABP_Raw']
            age_l, ht_l, wt_l, gd_l = (wins.get(k, np.nan)
                for k in ['Age', 'Height', 'Weight', 'Gender'])
            subj_hash = abs(hash(fname)) % (10**8)

            for w in range(min(len(ppg_list), len(abp_list))):
                ppg_w = np.squeeze(ppg_list[w])
                abp_w = np.squeeze(abp_list[w])
                if ppg_w.size < FS_PULSE*5 or abp_w.size < FS_PULSE*5:
                    continue
                ml = min(len(ppg_w), len(abp_w))
                ppg_w, abp_w = ppg_w[:ml], abp_w[:ml]

                try:
                    valleys, _ = find_peaks(
                        -savgol_filter(ppg_w, 11, 3),
                        distance=int(FS_PULSE*MIN_BEAT_S), prominence=0.05)
                except Exception:
                    continue
                if len(valleys) < 2: track["peak"] += 1; continue

                for j in range(len(valleys) - 1):
                    s, e   = valleys[j], valleys[j+1]
                    beat_s = (e - s) / FS_PULSE
                    if not (MIN_BEAT_S <= beat_s <= MAX_BEAT_S): continue
                    beat_abp = abp_w[s:e]
                    sbp, dbp = float(beat_abp.max()), float(beat_abp.min())
                    if not qc_bp(sbp, dbp): track["qc"] += 1; continue

                    ppg_r = normalize_ppg(resample(ppg_w[s:e], FIXED_LEN))
                    abp_r = normalize_abp_population(resample(beat_abp, FIXED_LEN))
                    out_ppg.append(ppg_r.astype(np.float32))
                    out_abp.append(abp_r)
                    out_bp.append([sbp, dbp])
                    out_dur.append(np.float32(beat_s))
                    out_subj.append(np.int32(subj_hash))
                    out_age.append(np.float32(_parse(age_l, w)))
                    out_gender.append(np.float32(_parse(gd_l, w, True)))
                    out_height.append(np.float32(_parse(ht_l, w)))
                    out_weight.append(np.float32(_parse(wt_l, w)))
                    track["ok"] += 1
        except Exception:
            track["err"] += 1

        if (fi + 1) % SAVE_EVERY_N == 0 or fi == len(todo) - 1:
            _flush(fi == len(todo) - 1)
            print(f"  [{fi+1}/{len(todo)}]  ok={track['ok']:,}  "
                  f"qc={track['qc']:,}  peak={track['peak']:,}  err={track['err']}")

    print("\n  PulseDB extraction complete!")

# ============================================================
# EXECUTION
# ============================================================
if __name__ == "__main__":
    print(f"\n{'='*55}")
    print(f"  Phase 1 FIXED — pop normalization")
    print(f"  ABP stored = (mmHg − {POP_MU}) / {POP_SIG}")
    print(f"  Phase 3 denorm: mmHg = stored × {POP_SIG} + {POP_MU}")
    print(f"{'='*55}")

    extract_uci()
    extract_hospital()
    extract_sensors()
    extract_bcg()
    extract_pulsedb()

    print("\n  Phase 1 FIXED Complete!")

In [ ]:
import os
import glob
import numpy as np
import gc

# ============================================================
# 1. CONFIGURATION
# ============================================================
CHUNKS_DIR = "outputs/phase_1/data/pulsedb_chunks"
OUT_FILE = "outputs/phase_1/data/pulsedb_tensors.npz"

def concatenate_pulsedb_chunks():
    print("="*65)
    print("PHASE 1: PulseDB concatenator (merging chunk files)")
    print("="*65)

    chunk_files = sorted(glob.glob(os.path.join(CHUNKS_DIR, "pulsedb_part_*.npz")))
    
    if not chunk_files:
        print(f"[ERROR] no chunk files found in {CHUNKS_DIR}")
        return

    print(f"[INFO] found {len(chunk_files)} chunk files")
    print("[INFO] loading and merging... (this may take a while)")

    all_X_ppg = []
    all_Y_abp = []
    all_Y_bp = []
    all_dur = []
    all_subj = []
    all_age, all_gender, all_height, all_weight = [], [], [], []

    total_beats = 0

    for i, f in enumerate(chunk_files):
        data = np.load(f)
        
        all_X_ppg.append(data['X_ppg'])
        all_Y_abp.append(data['Y_abp'])
        all_Y_bp.append(data['Y_bp'])
        all_dur.append(data['beat_durations'])
        all_subj.append(data['subject_ids'])
        
        all_age.append(data['age'])
        all_gender.append(data['gender'])
        all_height.append(data['height'])
        all_weight.append(data['weight'])
        
        beats_in_chunk = len(data['X_ppg'])
        total_beats += beats_in_chunk
        
        if (i + 1) % 5 == 0 or (i + 1) == len(chunk_files):
            print(f"  -> loaded {i + 1}/{len(chunk_files)} files | {total_beats:,} beats total")

    print("\n[INFO] all files loaded; concatenating tensors...")
    
    X_ppg_cat = np.concatenate(all_X_ppg, axis=0)
    Y_abp_cat = np.concatenate(all_Y_abp, axis=0)
    Y_bp_cat = np.concatenate(all_Y_bp, axis=0)
    dur_cat = np.concatenate(all_dur, axis=0)
    subj_cat = np.concatenate(all_subj, axis=0)
    
    age_cat = np.concatenate(all_age, axis=0)
    gender_cat = np.concatenate(all_gender, axis=0)
    height_cat = np.concatenate(all_height, axis=0)
    weight_cat = np.concatenate(all_weight, axis=0)

    del all_X_ppg, all_Y_abp, all_Y_bp, all_dur, all_subj
    del all_age, all_gender, all_height, all_weight
    gc.collect()

    print("\n" + "="*55)
    print("📊 FINAL TENSOR SHAPES (PulseDB)")
    print("="*55)
    print(f"X_ppg          : {X_ppg_cat.shape}")
    print(f"Y_abp          : {Y_abp_cat.shape}")
    print(f"Y_bp (SBP/DBP) : {Y_bp_cat.shape}")
    print(f"Demographics   : Age{age_cat.shape}, Gender{gender_cat.shape}")
    print("="*55)

    print("\n[INFO] saving to a single master file...")
    np.savez_compressed(
        OUT_FILE,
        X_ppg=X_ppg_cat, 
        Y_abp=Y_abp_cat, 
        Y_bp=Y_bp_cat,   
        beat_durations=dur_cat, 
        subject_ids=subj_cat,
        age=age_cat,
        gender=gender_cat,
        height=height_cat,
        weight=weight_cat
    )
    
    print(f"Done. Merged file saved to: {OUT_FILE}")
    print("Tip: the pulsedb_chunks folder can now be deleted to reclaim disk space.")

if __name__ == "__main__":
    concatenate_pulsedb_chunks()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

# ============================================================
# 1) LOAD PHASE 1 TENSORS (The processed beats)
# ============================================================
print("[INFO] Loading Phase 1 Tensors for all datasets...")
uci_p1 = np.load("outputs/phase_1/data/universal_tensors.npz")
hosp_p1 = np.load("outputs/phase_1/data/hospital_tensors.npz")
sens_p1 = np.load("outputs/phase_1/data/sensors_tensors.npz")
bcg_p1 = np.load("outputs/phase_1/data/bcg_tensors.npz")
puls_p1 = np.load("outputs/phase_1/data/pulsedb_tensors.npz")

uci_subs = uci_p1["subject_ids"]
hosp_subs = hosp_p1["subject_ids"]
sens_subs = sens_p1["subject_ids"]
bcg_subs = bcg_p1["subject_ids"]
puls_subs = puls_p1["subject_ids"]

# ============================================================
# 2) DATASET ISOLATION (Add Prefixes to prevent ID collision)
# ============================================================
uci_unique_ids = [f"UCI_{s}" for s in np.unique(uci_subs)]
hosp_unique_ids = [f"HOSP_{s}" for s in np.unique(hosp_subs)]
sens_unique_ids = [f"SENS_{s}" for s in np.unique(sens_subs)]
bcg_unique_ids = [f"BCG_{s}" for s in np.unique(bcg_subs)]
puls_unique_ids = [f"PULS_{s}" for s in np.unique(puls_subs)]

# Create DataFrames for analysis
df_uci = pd.DataFrame({"full_id": [f"UCI_{s}" for s in uci_subs]})
df_hosp = pd.DataFrame({"full_id": [f"HOSP_{s}" for s in hosp_subs]})
df_sens = pd.DataFrame({"full_id": [f"SENS_{s}" for s in sens_subs]})
df_bcg = pd.DataFrame({"full_id": [f"BCG_{s}" for s in bcg_subs]})
df_puls = pd.DataFrame({"full_id": [f"PULS_{s}" for s in puls_subs]})

# ============================================================
# 3) LEAKAGE & OVERLAP ANALYSIS
# ============================================================
all_sets = [set(uci_unique_ids), set(hosp_unique_ids), set(sens_unique_ids), set(bcg_unique_ids), set(puls_unique_ids)]
dataset_names = ["UCI", "Hospital", "Sensors", "BCG", "PulseDB"]
dfs = [df_uci, df_hosp, df_sens, df_bcg, df_puls]

print("\n" + "="*50)
print(f"{'SUBJECT DISTRIBUTION & LEAKAGE CHECK':^50}")
print("="*50)

for name, unique_set, df in zip(dataset_names, all_sets, dfs):
    print(f"{name:10s} Dataset:")
    print(f"  - Total Beats     : {len(df):,}")
    print(f"  - Unique Subjects : {len(unique_set):,}")
    avg_beats = len(df)/len(unique_set) if len(unique_set) > 0 else 0
    print(f"  - Avg Beats/Subj  : {avg_beats:.1f}\n")

# Check cross-contamination across all sets
overlap_found = False
print(f"Cross-Dataset Contamination Check:")
for i in range(len(all_sets)):
    for j in range(i + 1, len(all_sets)):
        overlap = all_sets[i].intersection(all_sets[j])
        if len(overlap) > 0:
            print(f"  🚨 WARNING: Found {len(overlap)} overlapping IDs between {dataset_names[i]} and {dataset_names[j]}!")
            overlap_found = True

if not overlap_found:
    print("  ✅ SUCCESS: All datasets are perfectly isolated from each other.")

# ============================================================
# 4) VISUALIZATION: BEATS PER SUBJECT
# ============================================================
fig, ax = plt.subplots(2, 3, figsize=(18, 10))
ax = ax.flatten()

plot_configs = [
    (df_uci, "UCI: Beats per Subject", "skyblue"),
    (df_hosp, "Hospital: Beats per Subject", "salmon"),
    (df_sens, "Sensors: Beats per Subject", "lightgreen"),
    (df_bcg, "BCG: Beats per Subject", "orchid"),
    (df_puls, "PulseDB: Beats per Subject", "orange")
]

for i, (df, title, color) in enumerate(plot_configs):
    if len(df) == 0: continue
    counts = df["full_id"].value_counts()
    sns.histplot(counts, bins=30, ax=ax[i], color=color)
    ax[i].set_title(title)
    ax[i].set_xlabel("Number of Beats")
    ax[i].set_ylabel("Count")

ax[5].axis('off')

plt.tight_layout()
plt.show()

# ============================================================
# 5) DATA SPLIT STRATEGY (Subject-Level)
# ============================================================
print("\n" + "="*50)
print("[STRATEGY] Subject-level Split Verification Example (UCI):")
print("="*50)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(uci_p1["X_ppg"], groups=uci_p1["subject_ids"]))

train_subs = set(uci_p1["subject_ids"][train_idx])
test_subs = set(uci_p1["subject_ids"][test_idx])
leakage = train_subs.intersection(test_subs)

print(f"  - Train Subjects: {len(train_subs):,}")
print(f"  - Test Subjects : {len(test_subs):,}")
print(f"  - Leakage       : {len(leakage)} (Must be 0)")
if len(leakage) == 0:
     print("  ✅ SPLIT SUCCESS: No data leakage between train and test sets.")

In [ ]:
import os
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# ============================================================
# make_pre_split()
# Source : universal_tensors_clean.npz  (245,666 beats, 11,984 subjects)
# Outputs: outputs/phase_1/data/pre_split/
#            train_split.npz  val_split.npz  test_split.npz
#            split_metadata.npz
# Window  : 11 beats (half=5, stride=3)
# Split   : subject-level GroupShuffleSplit — test=15%, val≈15%
# ============================================================

def make_pre_split(
    src_path   = "outputs/phase_1/data/universal_tensors_clean.npz",
    out_dir    = "outputs/phase_1/data/pre_split",
    n_beats    = 10,
    window_stride = 3,
    test_frac  = 0.15,
    random_state = 42,
):
    os.makedirs(out_dir, exist_ok=True)

    # ── 1. Load raw beats ──────────────────────────────────────────────────
    print(f"[make_pre_split] Loading {src_path} ...")
    raw      = np.load(src_path)
    X_ppg    = raw["X_ppg"]          # (N, 1, 128)
    Y_abp    = raw["Y_abp"]          # (N, 128)  pop-normalised
    Y_bp     = raw["Y_bp"]           # (N, 2)    [SBP, DBP] mmHg
    dur_all  = raw["beat_durations"] # (N,)      seconds
    subj_raw = raw["subject_ids"]    # (N,)      original UCI subject IDs
    pop_mu   = float(raw["pop_mu"])
    pop_sig  = float(raw["pop_sig"])
    N        = len(X_ppg)
    print(f"  beats={N:,}   pop_mu={pop_mu}   pop_sig={pop_sig}")

    # ── 2. Subject re-indexing: original IDs → 0..n_subjects-1 ───────────
    unique_subjs = np.unique(subj_raw)         # sorted
    n_subjects   = len(unique_subjs)
    subj_remap   = {s: i for i, s in enumerate(unique_subjs)}
    subj_idx     = np.array([subj_remap[s] for s in subj_raw], dtype=np.int64)
    print(f"  n_subjects={n_subjects:,}   subj range 0..{n_subjects-1}")

    # ── 3. Q proxy: rectified gradient, integral-normalised to sum≈8 ──────
    print("  Computing Q proxy ...")
    ppg = X_ppg[:, 0, :]                       # (N, 128)
    d   = np.gradient(ppg, axis=-1)
    q   = np.maximum(d, 0.0)
    a   = np.trapezoid(q, axis=-1)[..., np.newaxis]  # (N, 1)
    a   = np.where(a < 1e-6, 1e-6, a)
    Q_all = (q / a * 8.0).astype(np.float32)   # (N, 128)

    # ── 4. Per-subject baseline: MAP of first beat (pop-norm space) ───────
    sort_order   = np.argsort(subj_raw, kind='stable')
    sorted_subjs = subj_raw[sort_order]
    first_pos    = np.searchsorted(sorted_subjs, unique_subjs)
    first_idx    = sort_order[first_pos]
    map_vals     = (Y_abp[first_idx].max(axis=-1) + Y_abp[first_idx].min(axis=-1)) / 2.0
    subj_to_base = dict(zip(unique_subjs, map_vals.astype(np.float32)))
    base_all     = np.array([subj_to_base[s] for s in subj_raw], dtype=np.float32)

    # ── 5. HTN labels ─────────────────────────────────────────────────────
    sbp_all = Y_bp[:, 0]
    dbp_all = Y_bp[:, 1]
    htn_all = ((sbp_all >= 130) | (dbp_all >= 80)).astype(np.float32)

    # Domain label: all UCI → 0
    dom_all = np.zeros(N, dtype=np.int64)

    # ── 6. Subject-level train / val / test split (on BEATS, not windows) ─
    print("  Splitting subjects ...")
    gss = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=random_state)
    tr_val_idx, te_idx = next(gss.split(X_ppg, groups=subj_raw))

    val_frac_of_remainder = test_frac / (1.0 - test_frac)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_remainder, random_state=random_state)
    _tr, _val = next(gss2.split(X_ppg[tr_val_idx], groups=subj_raw[tr_val_idx]))
    tr_idx = tr_val_idx[_tr]
    va_idx = tr_val_idx[_val]

    # Subject counts per split
    n_tr  = len(np.unique(subj_raw[tr_idx]))
    n_va  = len(np.unique(subj_raw[va_idx]))
    n_te  = len(np.unique(subj_raw[te_idx]))
    print(f"  Subject counts — train:{n_tr}  val:{n_va}  test:{n_te}  (total={n_tr+n_va+n_te})")

    # ── 7. Build overlapping 11-beat windows ──────────────────────────────
    half = n_beats // 2  # 5

    def _build_windows(beat_idx):
        idx_s    = np.sort(beat_idx)
        X_s      = X_ppg[idx_s, 0, :]   # (M, 128) — squeeze channel dim
        Q_s      = Q_all[idx_s]          # (M, 128)
        Yabp_s   = Y_abp[idx_s]          # (M, 128)
        Ybp_s    = Y_bp[idx_s]           # (M, 2)
        dur_s    = dur_all[idx_s]        # (M,)
        htn_s    = htn_all[idx_s]        # (M,)
        subj_s   = subj_idx[idx_s]       # (M,)  re-indexed 0..n_subjects-1
        dom_s    = dom_all[idx_s]        # (M,)
        base_s   = base_all[idx_s]       # (M,)

        Xw, Qw, Yaw, Ybpw = [], [], [], []
        durw, htnw, subjw, domw, basew = [], [], [], [], []

        M = len(X_s)
        for i in range(half, M - half, window_stride):
            if subj_s[i - half] != subj_s[i + half]:
                continue  # window crosses subject boundary
            sl = slice(i - half, i + half + 1)
            Xw.append(X_s[sl])
            Qw.append(Q_s[sl])
            Yaw.append(Yabp_s[sl])
            Ybpw.append(Ybp_s[i])
            durw.append(dur_s[i])
            htnw.append(htn_s[i])
            subjw.append(subj_s[i])
            domw.append(dom_s[i])
            basew.append(base_s[i])

        return (
            np.array(Xw,    dtype=np.float32),  # (W, 11, 128)
            np.array(Qw,    dtype=np.float32),  # (W, 11, 128)
            np.array(Yaw,   dtype=np.float32),  # (W, 11, 128)
            np.array(Ybpw,  dtype=np.float32),  # (W, 2)
            np.array(durw,  dtype=np.float32),  # (W,)
            np.array(htnw,  dtype=np.float32),  # (W,)
            np.array(subjw, dtype=np.int64),    # (W,)
            np.array(domw,  dtype=np.int64),    # (W,)
            np.array(basew, dtype=np.float32),  # (W,)
        )

    print("  Building train windows ...")
    tr_out = _build_windows(tr_idx)
    print(f"    train windows: {len(tr_out[0]):,}")

    print("  Building val windows ...")
    va_out = _build_windows(va_idx)
    print(f"    val   windows: {len(va_out[0]):,}")

    print("  Building test windows ...")
    te_out = _build_windows(te_idx)
    print(f"    test  windows: {len(te_out[0]):,}")

    # ── 8. Save ───────────────────────────────────────────────────────────
    keys = ["X", "Q", "Yabp", "Ybp", "dur", "htn", "subj", "dom", "base"]

    def _save(name, arrays):
        path = os.path.join(out_dir, f"{name}_split.npz")
        np.savez_compressed(path, **dict(zip(keys, arrays)))
        print(f"  Saved {path}  X={arrays[0].shape}")

    _save("train", tr_out)
    _save("val",   va_out)
    _save("test",  te_out)

    # Metadata: SBP/DBP stats from TRAIN beats only (no leakage)
    sbp_tr = sbp_all[tr_idx]
    dbp_tr = dbp_all[tr_idx]
    np.savez_compressed(
        os.path.join(out_dir, "split_metadata.npz"),
        pop_mu     = np.float32(pop_mu),
        pop_sig    = np.float32(pop_sig),
        sbp_mu     = np.float32(sbp_tr.mean()),
        sbp_sig    = np.float32(sbp_tr.std()),
        dbp_mu     = np.float32(dbp_tr.mean()),
        dbp_sig    = np.float32(dbp_tr.std()),
        n_subjects = np.int64(n_subjects),
        n_beats    = np.int64(n_beats),
        window_stride = np.int64(window_stride),
        test_frac  = np.float32(test_frac),
        random_state = np.int64(random_state),
    )
    print(f"  Saved split_metadata.npz")
    print(f"\n[make_pre_split] Done.")
    print(f"  train:{len(tr_out[0]):,}  val:{len(va_out[0]):,}  test:{len(te_out[0]):,}  windows")
    print(f"  pop_mu={pop_mu}  pop_sig={pop_sig}")
    print(f"  sbp_mu={sbp_tr.mean():.1f}  sbp_sig={sbp_tr.std():.1f}")
    print(f"  dbp_mu={dbp_tr.mean():.1f}  dbp_sig={dbp_tr.std():.1f}")


# Run it
make_pre_split()

In [ ]:
import numpy as np
import pandas as pd
import os

files = {
    "UCI Dataset": "outputs/phase_1/data/universal_tensors.npz",
    "Hospital Dataset": "outputs/phase_1/data/hospital_tensors.npz",
    "Sensors Dataset": "outputs/phase_1/data/sensors_tensors.npz",
    "BCG Dataset": "outputs/phase_1/data/bcg_tensors.npz",
    "PulseDB Dataset": "outputs/phase_1/data/pulsedb_tensors.npz" 
}

summary_list = []

print("[INFO] Inspecting 5 Tensor files...\n")

for name, path in files.items():
    if os.path.exists(path):
        data = np.load(path)
        
        unique_subs = len(np.unique(data["subject_ids"]))
        
        summary = {
            "Dataset": name,
            "Total Beats (N)": f"{len(data['X_ppg']):,}",
            "X_ppg (Input)": str(data["X_ppg"].shape),
            "Y_abp (Waveform)": str(data["Y_abp"].shape) if "Y_abp" in data else "N/A",
            "Y_bp (Target)": str(data["Y_bp"].shape),
            "Unique Subjects": f"{unique_subs:,}",
            "Cross-Val Folds": "Yes (0-4)" if "fold_indices" in data else "No"
        }
        summary_list.append(summary)
    else:
        summary_list.append({
            "Dataset": name, 
            "Total Beats (N)": "FILE NOT FOUND",
            "X_ppg (Input)": "-", "Y_abp (Waveform)": "-", "Y_bp (Target)": "-", 
            "Unique Subjects": "-", "Cross-Val Folds": "-"
        })

df_summary = pd.DataFrame(summary_list)
df_summary.set_index("Dataset", inplace=True)

print("="*95)
print(f"{'📦 PHASE 1: ALL 5 DATA TENSORS SUMMARY TABLE':^95}")
print("="*95)
print(df_summary.to_string())
print("="*95)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wasserstein_distance

# ============================================================
# 0. STYLE & CONFIGURATION
# ============================================================
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    "font.family": "serif",
    "figure.dpi": 150,
    "axes.labelsize": 11,
    "axes.titlesize": 13
})

C = {
    "uci": "#1f77b4",     # Blue
    "hosp": "#d62728",    # Red
    "sensors": "#2ca02c", # Green
    "bcg": "#9467bd",     # Purple
    "puls": "#ff7f0e",    # Orange (PulseDB)
    "line": "#000000"     # Black for Threshold
}

# ============================================================
# 1. LOAD & PREPARE DATA
# ============================================================
print("[INFO] Loading Phase 1 Tensors for Domain Analysis...")
uci = np.load("outputs/phase_1/data/universal_tensors.npz")
hosp = np.load("outputs/phase_1/data/hospital_tensors.npz")
sens = np.load("outputs/phase_1/data/sensors_tensors.npz")
bcg = np.load("outputs/phase_1/data/bcg_tensors.npz")
puls = np.load("outputs/phase_1/data/pulsedb_tensors.npz")

datasets = {
    "UCI": {"data": uci, "color": C["uci"]},
    "Hospital": {"data": hosp, "color": C["hosp"]},
    "Sensors": {"data": sens, "color": C["sensors"]},
    "BCG": {"data": bcg, "color": C["bcg"]},
    "PulseDB": {"data": puls, "color": C["puls"]} 
}

# ============================================================
# Helper Functions
# ============================================================
def extract_bp(dataset_dict):
    y_bp = dataset_dict["data"]["Y_bp"]
    return y_bp[:, 0], y_bp[:, 1]

def get_std(dataset_dict):
    tensor = dataset_dict["data"]["X_ppg"]
    flat = tensor.squeeze() if tensor.ndim == 3 else tensor
    return np.std(flat, axis=1)

# ============================================================
# 2. OVERLAY VISUALIZATION (SBP & DBP)
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

for name, meta in datasets.items():
    sbp, dbp = extract_bp(meta)
    
    sns.kdeplot(sbp, fill=True, color=meta["color"], label=f"{name}", alpha=0.3, ax=ax[0], bw_adjust=1.5)
    sns.kdeplot(dbp, fill=True, color=meta["color"], label=f"{name}", alpha=0.3, ax=ax[1], bw_adjust=1.5)

ax[0].axvline(130, color=C["line"], ls="--", lw=1.5, label="HTN Threshold (130)")
ax[1].axvline(80, color=C["line"], ls="--", lw=1.5, label="HTN Threshold (80)")

ax[0].set_title("Distribution Overlay: SBP", fontweight="bold")
ax[0].set_xlabel("Systolic Blood Pressure (mmHg)")
ax[0].set_xlim([80, 200]) 
ax[0].legend()

ax[1].set_title("Distribution Overlay: DBP", fontweight="bold")
ax[1].set_xlabel("Diastolic Blood Pressure (mmHg)")
ax[1].set_xlim([40, 120])
ax[1].legend()

plt.tight_layout()
plt.show()

# ============================================================
# 3. DOMAIN SHIFT ANALYSIS (Signal Variability)
# ============================================================
plt.figure(figsize=(14, 6))
std_dict = {}

for name, meta in datasets.items():
    std_val = get_std(meta)
    std_dict[name] = std_val
    sns.kdeplot(std_val, fill=True, color=meta["color"], label=name, bw_adjust=1.5, alpha=0.2)

dist_hosp = wasserstein_distance(std_dict["UCI"], std_dict["Hospital"])
dist_sens = wasserstein_distance(std_dict["UCI"], std_dict["Sensors"])
dist_bcg = wasserstein_distance(std_dict["UCI"], std_dict["BCG"])
dist_puls = wasserstein_distance(std_dict["UCI"], std_dict["PulseDB"]) 

title_text = "Signal Variability (PPG Shape Distribution)\n"
title_text += f"[Domain Gap from UCI] Hosp: {dist_hosp:.4f} | Sens: {dist_sens:.4f} | BCG: {dist_bcg:.4f} | Puls: {dist_puls:.4f}"

plt.title(title_text, fontweight="bold")
plt.xlabel("Signal Standard Deviation (Normalized PPG Amplitude)")
plt.ylabel("Density")
plt.legend()
plt.show()

# ============================================================
# 4. PERFECT STAT SUMMARY (Pandas Table)
# ============================================================
stats_data = {"Metric": ["SBP Mean", "SBP Std", "DBP Mean", "DBP Std", "Signal Std Mean"]}

for name, meta in datasets.items():
    sbp, dbp = extract_bp(meta)
    std = get_std(meta)
    stats_data[name] = [
        sbp.mean(), sbp.std(), 
        dbp.mean(), dbp.std(), 
        std.mean()
    ]

df_stats = pd.DataFrame(stats_data).set_index("Metric").round(2)

print("\n" + "="*80)
print(f"{'DOMAIN SHIFT STATISTICAL SUMMARY (All 5 Datasets)':^80}")
print("="*80)
print(df_stats.to_string())
print("="*80)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wasserstein_distance
import os

# ============================================================
# 0. CONFIGURATION & STYLE
# ============================================================
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({"font.family": "serif", "figure.dpi": 120})

DATA_PATHS = {
    "UCI": "outputs/phase_1/data/universal_tensors.npz",
    "Hospital": "outputs/phase_1/data/hospital_tensors.npz",
    "Sensors": "outputs/phase_1/data/sensors_tensors.npz",
    "BCG": "outputs/phase_1/data/bcg_tensors.npz",
    "PulseDB": "outputs/phase_1/data/pulsedb_tensors.npz"
}

COLORS = {
    "UCI": "#1f77b4", 
    "Hospital": "#d62728", 
    "Sensors": "#2ca02c", 
    "BCG": "#9467bd",
    "PulseDB": "#ff7f0e"
}

def categorize_bp(sbp, dbp):
    if sbp < 120 and dbp < 80: return "Normal"
    elif 120 <= sbp < 130 and dbp < 80: return "Elevated"
    elif 130 <= sbp < 140 or 80 <= dbp < 90: return "Stage 1 HTN"
    elif sbp >= 140 or dbp >= 90: return "Stage 2 HTN"
    return "Crisis/Unknown"

# ============================================================
# 1. DATA INTEGRITY & QUALITY INSPECTION
# ============================================================
print(f"{'🔍 DATA INTEGRITY REPORT':^60}")
print("-" * 60)

quality_metrics = []

for name, path in DATA_PATHS.items():
    if not os.path.exists(path): continue
    data = np.load(path)
    x = data["X_ppg"]
    y_bp = data["Y_bp"]
    
    has_nan = np.isnan(x).any() or np.isnan(y_bp).any()
    ppg_min, ppg_max = x.min(), x.max()
    
    quality_metrics.append({
        "Dataset": name,
        "Has NaN/Inf": "❌ No" if not has_nan else "⚠️ Yes",
        "PPG Min": f"{ppg_min:.3f}",
        "PPG Max": f"{ppg_max:.3f}",
        "BP Samples": len(y_bp)
    })

print(pd.DataFrame(quality_metrics).to_string(index=False))

# ============================================================
# 2. MORPHOLOGY VISUALIZATION (Signal Comparison)
# ============================================================
print("\n[INFO] Generating Morphology Visualizations...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (name, path) in enumerate(DATA_PATHS.items()):
    if not os.path.exists(path): continue
    
    data = np.load(path)
    x = data["X_ppg"]
    
    indices = np.random.choice(len(x), 3, replace=False)
    
    for idx in indices:
        axes[i].plot(x[idx].squeeze(), alpha=0.6)
        
    axes[i].set_title(f"PPG Morphology: {name}", fontweight="bold", color=COLORS[name])
    axes[i].set_ylim([-0.1, 1.1])
    axes[i].set_xlabel("Time Steps (128)")
    
    if "Y_abp" in data and name != "Hospital":
        y_wave_pop = data["Y_abp"][indices[0]].squeeze()
        
        axes[i].plot(y_wave_pop, color='black', lw=2, linestyle='--', label="ABP (Pop Scaled)")
        
        axes[i].set_ylim([-2.0, 6.0])
        axes[i].legend(fontsize='small')

axes[5].axis('off')

plt.tight_layout()
plt.show()

# ============================================================
# 3. CLINICAL CLASS DISTRIBUTION (AHA/ACC 2017)
# ============================================================
print("\n[INFO] Analyzing Blood Pressure Clinical Categories...")
all_dist = []

for name, path in DATA_PATHS.items():
    if not os.path.exists(path): continue
    data = np.load(path)
    y_bp = data["Y_bp"]
    
    categories = [categorize_bp(s, d) for s, d in y_bp]
    counts = pd.Series(categories).value_counts(normalize=True) * 100
    
    for cat, val in counts.items():
        all_dist.append({"Dataset": name, "Category": cat, "Percentage": val})

df_dist = pd.DataFrame(all_dist)

plt.figure(figsize=(14, 6))
sns.barplot(data=df_dist, x="Category", y="Percentage", hue="Dataset", palette=COLORS)
plt.title("Clinical BP Category Distribution across Domains", fontweight="bold")
plt.ylabel("Percentage of Samples (%)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# ============================================================
# 4. FINAL CROSS-DOMAIN GAP SUMMARY (Wasserstein Distance)
# ============================================================
print("\n" + "="*60)
print(f"{'FINAL DOMAIN GAP ANALYSIS (Ref: UCI)':^60}")
print("="*60)

uci_std = np.std(np.load(DATA_PATHS["UCI"])["X_ppg"].squeeze(), axis=1)
gap_results = []

for name, path in DATA_PATHS.items():
    if name == "UCI" or not os.path.exists(path): continue
    curr_std = np.std(np.load(path)["X_ppg"].squeeze(), axis=1)
    dist = wasserstein_distance(uci_std, curr_std)
    gap_results.append({"Target Domain": name, "Wasserstein Distance": f"{dist:.4f}"})

print(pd.DataFrame(gap_results).to_string(index=False))
print("-" * 60)